In [ ]:
# Mount Drive & Install Tokenizer
from google.colab import drive
drive.mount('/content/drive')
!pip install -q tokenizers

import json, math
import pandas as pd
import numpy as np
from collections import defaultdict
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Dropout, Concatenate
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

# Load dataset
file_path = '/content/drive/MyDrive/review-South_Carolina_10.json'  # 🔁 update path if needed
data = []
with open(file_path, 'r') as f:
    for line in f:
        try:
            item = json.loads(line)
            data.append({
                'user_id': item['user_id'],
                'gmap_id': item['gmap_id'],
                'rating': item['rating'],
                'text': item.get('text', ''),
                'time': item['time']
            })
        except:
            continue
df = pd.DataFrame(data)

# Encode user/item
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
df['user_idx'] = user_encoder.fit_transform(df['user_id'])
df['item_idx'] = item_encoder.fit_transform(df['gmap_id'])

# Tokenize text using Hugging Face `tokenizers`
tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
trainer = WordLevelTrainer(special_tokens=["[UNK]", "[PAD]"], vocab_size=10000)
tokenizer.pre_tokenizer = Whitespace()
texts = df['text'].astype(str).tolist()
tokenizer.train_from_iterator(texts, trainer=trainer)
df['tokens'] = [tokenizer.encode(text).ids for text in texts]

# Pad token sequences
MAX_LEN = 100
df['padded'] = pad_sequences(df['tokens'], maxlen=MAX_LEN, padding='post', truncating='post').tolist()

# Normalize time
scaler = MinMaxScaler()
df['time_norm'] = scaler.fit_transform(df[['time']])

# Chronological split
df = df.sort_values(by='time').reset_index(drop=True)
split_idx = int(0.8 * len(df))
train, test = df.iloc[:split_idx], df.iloc[split_idx:]

X_train_user = train['user_idx'].values
X_train_item = train['item_idx'].values
X_train_text = np.stack(train['padded'].values)
X_train_time = train[['time_norm']].values
y_train = train['rating'].values

X_test_user = test['user_idx'].values
X_test_item = test['item_idx'].values
X_test_text = np.stack(test['padded'].values)
X_test_time = test[['time_norm']].values
y_test = test['rating'].values

# Model
n_users = df['user_idx'].nunique()
n_items = df['item_idx'].nunique()
vocab_size = 10000
embedding_dim = 32

user_input = Input(shape=(), dtype='int32')
item_input = Input(shape=(), dtype='int32')
text_input = Input(shape=(MAX_LEN,), dtype='int32')
time_input = Input(shape=(1,), dtype='float32')

user_emb = Embedding(n_users, embedding_dim)(user_input)
item_emb = Embedding(n_items, embedding_dim)(item_input)
text_emb = Embedding(vocab_size, embedding_dim)(text_input)
text_flat = Flatten()(text_emb)

x = Concatenate()([
    Flatten()(user_emb),
    Flatten()(item_emb),
    text_flat,
    time_input
])
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(1, activation='linear')(x)

model = Model(inputs=[user_input, item_input, text_input, time_input], outputs=output)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
model.fit([X_train_user, X_train_item, X_train_text, X_train_time], y_train,
          validation_split=0.2, epochs=5, batch_size=128, callbacks=[early_stop])

# Evaluate
loss, mae = model.evaluate([X_test_user, X_test_item, X_test_text, X_test_time], y_test)
print(f"\nTest MAE: {mae:.4f}")



